# **🇹🇭 Super AI Engineer SS6: Constituency OCR with Qwen2.5-VL-7B**

This solution utilizes the robust and highly stable **Qwen2.5-VL-7B-Instruct** model running entirely offline on the Kaggle GPU.
It implements strict 4-bit quantization and aggressive VRAM garbage collection to ensure it fits perfectly within the 16GB limit of a Kaggle GPU (T4 / P100) without crashing.

### Kaggle Setup Instructions:
1. Ensure **GPU P100** or **GPU T4 x2** is enabled in the Kaggle Notebook settings.
2. Run the `pip install` cell with internet enabled during the first run.
3. Be sure to check that the dataset paths match your Kaggle environment (e.g., `/kaggle/input/superai-ss6-constituency`).

In [ ]:
# Install dependencies for Qwen2.5-VL and 4-bit quantization
!pip install -q git+https://github.com/huggingface/transformers accelerate bitsandbytes qwen-vl-utils rapidfuzz textdistance opencv-python pandas pillow torchvision

In [ ]:
import os
import glob
import json
import time
import re
import torch
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from rapidfuzz import process, fuzz
import textdistance
import gc

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

## 1. Load Submission Template & Utilities


In [ ]:
DATA_DIR = './data'     # Adjust this to '/kaggle/input/...' if running in Kaggle environment
IMG_DIR = os.path.join(DATA_DIR, 'images')
TEMPLATE_PATH = os.path.join(DATA_DIR, 'submission_template.csv')

sub_df = pd.read_csv(TEMPLATE_PATH)
doc_ids = sub_df['doc_id'].unique()
print(f"Loaded {len(sub_df)} rows for {len(doc_ids)} unique documents.")

def get_images_for_doc(doc_id):
    """Reads all PNG pages for a given doc_id."""
    images = []
    base_img = os.path.join(IMG_DIR, f"{doc_id}.png")
    if os.path.exists(base_img):
        images.append(base_img)
    
    page = 2
    while True:
        page_img = os.path.join(IMG_DIR, f"{doc_id}_page{page}.png")
        if os.path.exists(page_img):
            images.append(page_img)
            page += 1
        else:
            break
    return [Image.open(img) for img in images]

def clean_vote_string(s):
    """Removes non-digits and converts Thai numerals to Arabic."""
    thai_to_arabic = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
    s = str(s).translate(thai_to_arabic)
    s = re.sub(r'\D', '', s) # Keep only digits
    return s if s else "0"

## 2. Load Model into GPU with 4-Bit Precision
Using BitsAndBytes to reduce 7B model memory footprint from ~14GB to ~5GB, leaving plenty of room for processing high-resolution document context.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Loading Model in 4-bit config...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, 
    quantization_config=quantization_config,
    device_map="auto"
)
print("Model loaded successfully!")

## 3. Extraction Function (Strict VRAM Protection)
Processes large document images while constantly releasing tensor references to prevent Out-Of-Memory.

In [ ]:
def extract_votes_hf(doc_id, parties_list):
    images = get_images_for_doc(doc_id)
    if not images:
        return {p: "0" for p in parties_list}
    
    parties_str = "\n".join([f"- {p}" for p in parties_list])
    final_extracted_map = {p: "0" for p in parties_list}
    
    for page_idx, img in enumerate(images):
        # Enforce memory constraint on image size to prevent VRAM spikes
        content_blocks = [
            {"type": "image", "image": img, "max_pixels": 768 * 768},
            {"type": "text", "text": f"""
You are an exact Data Extraction AI analyzing Thai election voting result documents.
Find the vote count for the following political parties on this page:
{parties_str}

Rules:
1. Strict JSON format REQUIRED. Output must start with ```json and end with ```
2. If a party is NOT found on this page, DO NOT include it in the JSON.
3. Convert Thai numbers to Arabic numerals.
4. Output only the mapping of 'Party Name' -> 'Vote Count'.
"""}
        ]
        
        messages = [{"role": "user", "content": content_blocks}]
        text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text_prompt],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda")
        
        try:
            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs, 
                    max_new_tokens=512,
                    do_sample=False
                )
                
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            output_text = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]
        except Exception as e:
            print(f"Model Error on {doc_id} Page {page_idx+1}: {e}")
            output_text = ""
            
        # --- STRICT CLEANUP SECTION ---
        del inputs, text_prompt, messages, content_blocks
        if 'image_inputs' in locals(): del image_inputs
        if 'generated_ids' in locals(): del generated_ids
        if 'generated_ids_trimmed' in locals(): del generated_ids_trimmed
        torch.cuda.empty_cache()
        gc.collect()
        # ------------------------------
        
        # Parse JSON output robustly
        if output_text:
            try:
                # Regex to find JSON block even if model chatted a bit
                json_match = re.search(r'\{[^{}]*\}', output_text)
                if "```json" in output_text:
                    json_str = output_text.split("```json")[1].split("```")[0]
                elif json_match:
                    json_str = json_match.group(0)
                else:
                    json_str = output_text
                
                extracted_map = json.loads(json_str)
                for k, v in extracted_map.items():
                    v_clean = clean_vote_string(str(v))
                    if v_clean != "0":
                        final_extracted_map[k] = v_clean
            except Exception:
                pass # Ignore malformed json for this page
                
    return final_extracted_map

## 4. Evaluation Loop
Run this against the `sample_labels` to check your Levenshtein score.

In [ ]:
def evaluate_on_samples():
    label_files = glob.glob(os.path.join(DATA_DIR, "sample_labels", "*.json"))
    total_dist = 0
    total_rows = 0
    
    print(f"Evaluating on {len(label_files)} sample ground truths...")
    
    for lpath in label_files:
        with open(lpath, 'r', encoding='utf-8') as f:
            gt_data = json.load(f)
            
        doc_id = os.path.basename(lpath).replace('.json', '')
        gt_map = {item['party']: str(item['votes']) for item in gt_data['results']}
        
        pred_map = extract_votes_hf(doc_id, list(gt_map.keys()))
        
        for party, gt_votes in gt_map.items():
            pred_votes = pred_map.get(party, "0")
            
            if party not in pred_map and len(pred_map) > 0:
                 best_match = process.extractOne(party, list(pred_map.keys()), scorer=fuzz.ratio)
                 if best_match and best_match[1] > 75:
                     pred_votes = pred_map[best_match[0]]
            
            dist = textdistance.levenshtein(gt_votes, pred_votes)
            total_dist += dist
            total_rows += 1
            
    if total_rows > 0:
        print(f"Mean Levenshtein Distance: {total_dist / total_rows:.4f}")
        
# evaluate_on_samples() # Uncomment to run evaluation

## 5. Main Prediction Pipeline
Execute the final run on all test documents and output to Kaggle `submission.csv`.

In [ ]:
results_list = []

# Remove [:5] to run on the entire dataset.
for doc_id in tqdm(doc_ids[:5], desc="Processing Test Documents"):
    parties_to_find = sub_df[sub_df['doc_id'] == doc_id]['party_name'].tolist()
    
    extracted_map = extract_votes_hf(doc_id, parties_to_find)
    
    for party in parties_to_find:
        votes = extracted_map.get(party, "0")
        
        if party not in extracted_map and len(extracted_map) > 0:
             best_match = process.extractOne(party, list(extracted_map.keys()), scorer=fuzz.ratio)
             if best_match and best_match[1] > 75:
                 votes = extracted_map[best_match[0]]
                 
        results_list.append({
            'doc_id': doc_id,
            'party_name': party,
            'votes': clean_vote_string(votes)
        })
        
# Update the template dataframe
for res in results_list:
    mask = (sub_df['doc_id'] == res['doc_id']) & (sub_df['party_name'] == res['party_name'])
    sub_df.loc[mask, 'votes'] = res['votes']

sub_df.to_csv('submission.csv', index=False)
print("Submission successfully saved to submission.csv!")
sub_df.head(10)